In [1]:
!pip install -q xgboost

In [5]:
from google.colab import files

uploaded = files.upload()

Saving placement_predict_50k_adjusted (1).csv to placement_predict_50k_adjusted (1).csv


In [6]:
DATA_PATH = "placement_predict_50k_adjusted (1).csv"

In [7]:
import os

print(os.listdir("/content"))

['.config', 'placement_predict_50k_adjusted (1).csv', 'sample_data']


In [8]:
import glob
import os

csv_files = glob.glob("/content/*.csv")

print("CSV files found:")

for file in csv_files:
    print(file)

CSV files found:
/content/placement_predict_50k_adjusted (1).csv


In [9]:
import pandas as pd

DATA_PATH = csv_files[0]

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully!
Shape: (50000, 21)

Columns:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'PlacementStatus', 'IsAnomaly']

First 5 rows:


,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,CGPA,AttendancePercent,Internships,...,Workshops,Certifications,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,PlacementStatus,IsAnomaly
0,Female,Delhi,Tier3,IT,DataScience,Yes,Yes,6.63,68.3,2,...,0.0,1,0,62.3,6.57,40.6,65.7,No,0,0
1,Male,Chennai,Tier2,ECE,AI,Yes,No,6.40,71.0,1,...,0.0,2,0,44.0,5.86,40.3,51.8,No,0,0
2,Female,Hyderabad,Tier3,ECE,Networking,No,No,7.73,75.1,1,...,2.0,2,1,73.8,7.50,73.6,67.9,No,1,0
3,Female,Jaipur,Tier3,ECE,Embedded,No,No,9.73,99.2,4,...,5.0,6,2,100.0,9.41,98.7,NaN,No,1,0
4,Male,Ahmedabad,Tier3,Mechanical,DataScience,No,No,9.01,99.6,2,...,NaN,4,2,90.8,9.24,83.1,100.0,No,1,0


In [10]:
print("Target column:", "PlacementStatus" in df.columns)

print("\nPlacementStatus values:")
print(df["PlacementStatus"].value_counts())

Target column: True

PlacementStatus values:
PlacementStatus
0    26251
1    23749
Name: count, dtype: int64


In [11]:
# ============================================================
# ADA BOOST vs XGBOOST
# Placement Prediction Experiment
# Google Colab Version
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier


# ============================================================
# 1. SETTINGS
# ============================================================

RANDOM_STATE = 42
TARGET_COL = "PlacementStatus"


# ============================================================
# 2. LOAD DATASET
# ============================================================

import glob

csv_files = glob.glob("/content/*.csv")

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV file found. Please upload the placement dataset first."
    )

DATA_PATH = csv_files[0]

print("Loading dataset:")
print(DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("\nDataset shape:", df.shape)


# ============================================================
# 3. REMOVE ANOMALY COLUMN IF PRESENT
# ============================================================

if "IsAnomaly" in df.columns:
    df = df.drop(columns=["IsAnomaly"])
    print("Removed IsAnomaly column.")


# ============================================================
# 4. CREATE TARGET AND FEATURES
# ============================================================

y = df[TARGET_COL].astype(int)

X = df.drop(columns=[TARGET_COL])


# ============================================================
# 5. IDENTIFY CATEGORICAL AND NUMERIC COLUMNS
# ============================================================

cat_cols = X.select_dtypes(
    exclude="number"
).columns.tolist()

num_cols = X.select_dtypes(
    include="number"
).columns.tolist()

print("\nCategorical columns:")
print(cat_cols)

print("\nNumeric columns:")
print(num_cols)


# ============================================================
# 6. LABEL ENCODING
# ============================================================

encoders = {}

for c in cat_cols:

    le = LabelEncoder()

    X[c] = le.fit_transform(
        X[c].astype(str)
    )

    encoders[c] = le


print("\nCategorical encoding completed.")


# ============================================================
# 7. HANDLE MISSING VALUES
# ============================================================

imputer = SimpleImputer(
    strategy="median"
)

X[num_cols] = imputer.fit_transform(
    X[num_cols]
)

print("Missing values handled.")


# ============================================================
# 8. STANDARDIZE NUMERIC FEATURES
# ============================================================

scaler = StandardScaler()

X[num_cols] = scaler.fit_transform(
    X[num_cols]
)

print("Numeric features standardized.")


# ============================================================
# 9. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

VAL_SIZE = 0.15
TEST_SIZE = 0.15


# First create test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)


# Calculate validation ratio
val_ratio = VAL_SIZE / (1 - TEST_SIZE)


# Create train and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=val_ratio,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)


print("\n========================================")
print("DATA SPLIT")
print("========================================")

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


# ============================================================
# 10. RESULTS LIST
# ============================================================

results = []


# ============================================================
# 11. ADABOOST
# ============================================================

print("\n========================================")
print("TRAINING ADABOOST")
print("========================================")


ada_base = DecisionTreeClassifier(
    max_depth=2,
    random_state=RANDOM_STATE
)


ada = AdaBoostClassifier(
    estimator=ada_base,
    n_estimators=200,
    learning_rate=0.5,
    random_state=RANDOM_STATE
)


t0 = time.time()

ada.fit(
    X_train,
    y_train
)

ada_fit_time = time.time() - t0


# Predictions
ada_val_pred = ada.predict(X_val)

ada_val_proba = ada.predict_proba(
    X_val
)[:, 1]


# Metrics
ada_accuracy = accuracy_score(
    y_val,
    ada_val_pred
)

ada_f1 = f1_score(
    y_val,
    ada_val_pred
)

ada_auc = roc_auc_score(
    y_val,
    ada_val_proba
)


results.append({
    "model": "AdaBoost",
    "val_accuracy": ada_accuracy,
    "val_f1": ada_f1,
    "val_roc_auc": ada_auc,
    "best_n_estimators": ada.n_estimators,
    "fit_time_sec": round(ada_fit_time, 2)
})


print("AdaBoost completed!")
print("Accuracy:", round(ada_accuracy, 4))
print("F1:", round(ada_f1, 4))
print("ROC-AUC:", round(ada_auc, 4))
print("Training time:", round(ada_fit_time, 2), "seconds")


# ============================================================
# 12. XGBOOST
# ============================================================

print("\n========================================")
print("TRAINING XGBOOST")
print("========================================")


xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)


t0 = time.time()


xgb.fit(
    X_train,
    y_train,
    eval_set=[
        (X_val, y_val)
    ],
    verbose=False
)


xgb_fit_time = time.time() - t0


# Predictions
xgb_val_pred = xgb.predict(X_val)

xgb_val_proba = xgb.predict_proba(
    X_val
)[:, 1]


# Metrics
xgb_accuracy = accuracy_score(
    y_val,
    xgb_val_pred
)

xgb_f1 = f1_score(
    y_val,
    xgb_val_pred
)

xgb_auc = roc_auc_score(
    y_val,
    xgb_val_proba
)


results.append({
    "model": "XGBoost",
    "val_accuracy": xgb_accuracy,
    "val_f1": xgb_f1,
    "val_roc_auc": xgb_auc,
    "best_n_estimators": xgb.best_iteration + 1,
    "fit_time_sec": round(xgb_fit_time, 2)
})


print("XGBoost completed!")
print("Accuracy:", round(xgb_accuracy, 4))
print("F1:", round(xgb_f1, 4))
print("ROC-AUC:", round(xgb_auc, 4))
print("Trees used:", xgb.best_iteration + 1)
print("Training time:", round(xgb_fit_time, 2), "seconds")


# ============================================================
# 13. CREATE FINAL LEADERBOARD
# ============================================================

leaderboard = pd.DataFrame(
    results
).sort_values(
    "val_accuracy",
    ascending=False
).reset_index(drop=True)


print("\n\n========================================")
print("FINAL VALIDATION LEADERBOARD")
print("========================================")

display(leaderboard)


# ============================================================
# 14. SAVE RESULTS
# ============================================================

output_file = "/content/boosting_benchmark_results.csv"

leaderboard.to_csv(
    output_file,
    index=False
)

print("\nResults saved to:")
print(output_file)


# ============================================================
# 15. BEST MODEL
# ============================================================

best_model = leaderboard.iloc[0]["model"]

print("\n========================================")
print("BEST MODEL")
print("========================================")

print("Best model:", best_model)

Loading dataset:
/content/placement_predict_50k_adjusted (1).csv

Dataset shape: (50000, 21)
Removed IsAnomaly column.

Categorical columns:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'ExtraCurricular']

Numeric columns:
['CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore']

Categorical encoding completed.
Missing values handled.
Numeric features standardized.

DATA SPLIT
Train: (34999, 19)
Validation: (7501, 19)
Test: (7500, 19)

TRAINING ADABOOST
AdaBoost completed!
Accuracy: 0.7963
F1: 0.781
ROC-AUC: 0.8802
Training time: 14.56 seconds

TRAINING XGBOOST
XGBoost completed!
Accuracy: 0.7959
F1: 0.7827
ROC-AUC: 0.8826
Trees used: 135
Training time: 2.46 seconds


FINAL VALIDATION LEADERBOARD


,model,val_accuracy,val_f1,val_roc_auc,best_n_estimators,fit_time_sec
0,AdaBoost,0.796294,0.780963,0.880162,200,14.56
1,XGBoost,0.795894,0.782744,0.882638,135,2.46



Results saved to:
/content/boosting_benchmark_results.csv

BEST MODEL
Best model: AdaBoost


In [12]:
from google.colab import files

files.download("/content/boosting_benchmark_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>